In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_date, current_timestamp
import json
from pyspark.sql.types import *

KAFKA_BOOTSTRAP_SERVERS = 'kafka:9092'
KAFKA_TOPIC = 'facebook-post'
BRONZE_PATH = "hdfs://hadoop-namenode:8020/datalake/bronze/facebook4"


# Inicializar a sessão Spark
spark = SparkSession.builder \
    .appName("KafkaToBronzeLake") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.2") \
    .getOrCreate()

# Leitura do tópico Kafka
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

# Schema do JSON enviado
comment_schema = StructType([
    StructField("user", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", StringType(), True)
])

message_schema = StructType([
    StructField("id", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("post_content", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("likes", IntegerType(), True),
    StructField("shared", IntegerType(), True),
    StructField("comments", ArrayType(comment_schema), True)
])

df_parsed = df_raw.selectExpr("CAST(value AS STRING) as json") \
    .withColumn("data", from_json(col("json"), message_schema)) \
    .select("data.*")
df_parsed = df_parsed.withColumn("event_time", current_timestamp())

# Escrever os dados brutos na camada Bronze do Data Lake
query = (df_parsed.repartition(1).writeStream\
    .format("parquet")\
    .option("path", BRONZE_PATH)\
    .option("checkpointLocation", "hdfs://hadoop-namenode:8020/datalake/checkpoints/facebook_raw4")\
    .trigger(processingTime="10 minutes")\
    .outputMode("append")\
    .start())

query.awaitTermination()

Py4JJavaError: An error occurred while calling o94.start.
: org.apache.hadoop.hdfs.BlockMissingException: Could not obtain block: BP-578491989-172.18.0.5-1746996028330:blk_1073741922_1098 file=/datalake/checkpoints/facebook_raw4/metadata No live nodes contain current block Block locations: DatanodeInfoWithStorage[172.18.0.2:9866,DS-ee14ceaa-a7f5-471f-9812-96ca72c38470,DISK] Dead nodes:  DatanodeInfoWithStorage[172.18.0.2:9866,DS-ee14ceaa-a7f5-471f-9812-96ca72c38470,DISK]
	at org.apache.hadoop.hdfs.DFSInputStream.refetchLocations(DFSInputStream.java:1007)
	at org.apache.hadoop.hdfs.DFSInputStream.chooseDataNode(DFSInputStream.java:990)
	at org.apache.hadoop.hdfs.DFSInputStream.chooseDataNode(DFSInputStream.java:969)
	at org.apache.hadoop.hdfs.DFSInputStream.blockSeekTo(DFSInputStream.java:677)
	at org.apache.hadoop.hdfs.DFSInputStream.readWithStrategy(DFSInputStream.java:884)
	at org.apache.hadoop.hdfs.DFSInputStream.read(DFSInputStream.java:957)
	at java.base/java.io.DataInputStream.re

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col, to_timestamp
from datetime import datetime, timedelta
spark = SparkSession.builder \
    .appName("SocialMediaSilver") \
    .getOrCreate()


current_time = datetime.utcnow()
window_start = current_time - timedelta(minutes=10)
window_end = current_time

# Função para carregar e transformar cada fonte
def process_social_data(path, source):
    # Define a janela de 10 minutos
    df = spark.read.parquet(path).filter(
        (col("event_time") >= window_start.isoformat()) &
        (col("event_time") < window_end.isoformat())
    )
    if source == "facebook":
        return df.select(
            col("user_name").alias("author"),
            col("post_content").alias("content"),
            to_timestamp("created_at").alias("post_date"),
            col("likes").alias("likes"),
            col("comments").alias("comments"),
            col("shares").alias("shares"),
        ).withColumn("source", lit("facebook"))

    elif source == "instagram":
        return df.select(
            col("user_handle").alias("author"),
            col("caption").alias("content"),
            to_timestamp("posted_at").alias("post_date"),
            col("likes").alias("likes"),
            lit(None).alias("comments"),
            lit(None).cast("int").alias("shares")
        ).withColumn("source", lit("instagram"))

    elif source == "twitter":
        return df.select(
            col("user.screen_name").alias("author"),
            col("tweet.text").alias("content"),
            to_timestamp("tweet.created_at").alias("post_date"),
            col("metrics.likes").alias("likes"),
            col("metrics.replies").alias("comments"),
            col("metrics.retweets").alias("shares")
        ).withColumn("source", lit("twitter"))


# Paths da Bronze
facebook_df = process_social_data("hdfs://hadoop-namenode:8020/datalake/bronze/facebook", "facebook")
instagram_df = process_social_data("hdfs://hadoop-namenode:8020/datalake/bronze/instagram", "instagram")
# x_df = process_social_data("hdfs://hadoop-namenode:8020/datalake/bronze/x", "x")

# União e escrita na camada Silver
silver_df = facebook_df.unionByName(instagram_df)#.unionByName(x_df)

# silver_df.coalese(1).write.mode("append").parquet("hdfs://hadoop-namenode:8020/datalake/silver/social_media/")


In [7]:
silver_df.show(2000,False)

+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------